Floor area decrease

Type split favors apartment buildings over parcel and stuehus. Terraced housing relative increase as well in suburban compacting. 

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path



In [11]:
Floor_area = np.load(Path('Floor_area_TCJ.npz'))
Floor_area.files

['data', 'types', 'years']

In [9]:
#show floor area data
#get types
types = Floor_area['types']
print(types)

['Parcelhus' 'Rekkehus' 'Etagehus' 'Stuehus']


In [10]:
npz_path = Path('Floor_area_TCJ.npz')
Floor_area = np.load(npz_path, allow_pickle=True)

# inspect
print("keys:", Floor_area.files)
for k in Floor_area.files:
    print(k, np.asarray(Floor_area[k]).shape)

# choose main array (change 'data' if your key differs)
candidate_keys = [k for k in Floor_area.files if isinstance(Floor_area[k], np.ndarray)]
main_key = 'data' if 'data' in candidate_keys else max(candidate_keys, key=lambda k: np.asarray(Floor_area[k]).size)
data = np.asarray(Floor_area[main_key]).copy()   # mutable copy

types = np.asarray(Floor_area.get('types')) if 'types' in Floor_area.files else None
years = np.asarray(Floor_area.get('years')) if 'years' in Floor_area.files else None
if years is not None:
    years = years.astype(int)

# find Etagehus type index
type_name = 'Etagehus'
if types is not None:
    # case-insensitive match
    matches = [i for i, t in enumerate(types) if str(t).lower() == type_name.lower()]
    if not matches:
        raise ValueError(f"Type '{type_name}' not found in types: {types}")
    t_idx = matches[0]
else:
    raise ValueError("No 'types' in file — set t_idx manually")

# find year indices for 2050..2100
if years is None:
    raise ValueError("No 'years' in file — provide year indices manually")
mask = (years >= 2050) & (years <= 2100)
idxs = np.where(mask)[0]
if idxs.size == 0:
    raise ValueError("No years found in range 2050-2100 in years array")

# reference is 2050 value (must exist)
if 2050 in years:
    y2050 = int(np.where(years == 2050)[0][0])
else:
    raise ValueError("Year 2050 not found in years array")

# factors linearly from 1.0 (2050) to 0.4 (2100)
factors = np.linspace(1.0, 0.4, num=idxs.size)

# apply depending on dimensionality
if data.ndim == 2:   # (n_types, n_years)
    ref = data[t_idx, y2050]
    for k, y in enumerate(idxs):
        data[t_idx, y] = ref * factors[k]

elif data.ndim == 4:  # (ny, nx, n_types, n_years)
    ref_grid = data[:, :, t_idx, y2050].copy()
    for k, y in enumerate(idxs):
        data[:, :, t_idx, y] = ref_grid * factors[k]

else:
    raise ValueError(f"Unexpected data shape: {data.shape}")

# save backup + modified file
backup = npz_path.with_name(npz_path.stem + '_backup.npz')
Path(backup).write_bytes(Path(npz_path).read_bytes())
out = npz_path.with_name(npz_path.stem + '_modified.npz')
save_dict = {k: Floor_area[k] for k in Floor_area.files}
save_dict[main_key] = data
np.savez_compressed(out, **save_dict)
print("Saved modified Floor_area to", out)

keys: ['data', 'types', 'years']
data (501, 501, 4)
types (4,)
years (501,)


ValueError: Unexpected data shape: (501, 501, 4)

In [13]:
# ...existing code...
import numpy as np
from pathlib import Path

npz_path = Path('Floor_area_TCJ.npz')
F = np.load(npz_path, allow_pickle=True)

# detect main array
candidate_keys = [k for k in F.files if isinstance(F[k], np.ndarray)]
main_key = 'data' if 'data' in candidate_keys else max(candidate_keys, key=lambda k: np.asarray(F[k]).size)
data = np.asarray(F[main_key]).copy()

# get labels if present
types = np.asarray(F.get('types')) if 'types' in F.files else None
years = np.asarray(F.get('years')) if 'years' in F.files else None
if years is not None:
    years = years.astype(int)

print("original data.shape=", data.shape, "years=", None if years is None else years.shape)

# If data has no year axis, expand to (ny, nx, n_types, n_years)
if data.ndim == 3:   # (ny, nx, n_types) -> create time axis
    ny, nx, n_types = data.shape
    if years is None:
        years = np.arange(2050, 2101)   # create 2050..2100 if none provided
        print("Created years:", years[0], "->", years[-1])
    n_years = years.size
    data4 = np.repeat(data[..., np.newaxis], n_years, axis=3)  # copy baseline to all years
else:
    data4 = data.copy()
    if data4.ndim != 4:
        raise ValueError(f"Unhandled data shape: {data.shape}")

print("expanded data4.shape=", data4.shape)

# find type index
type_name = 'Etagehus'   # change to 'Etagehus' or your label if needed
if types is None:
    raise ValueError("No 'types' in file — set type index manually")
matches = [i for i, t in enumerate(types) if str(t).lower() == type_name.lower()]
if not matches:
    raise ValueError(f"Type '{type_name}' not found in types: {types}")
t_idx = matches[0]

# indices for target years 2050..2100
mask = (years >= 2050) & (years <= 2100)
idxs = np.where(mask)[0]
if idxs.size == 0:
    raise ValueError("No years found in range 2050-2100 in years array")
y2050 = int(np.where(years == 2050)[0][0])

# linear factors 1.0 -> 0.4
factors = np.linspace(1.0, 0.4, num=idxs.size)

# apply ramp (works for spatial grid)
ref_grid = data4[:, :, t_idx, y2050].copy()
for k, y in enumerate(idxs):
    data4[:, :, t_idx, y] = ref_grid * factors[k]

# save modified file (keep original backup)
backup = npz_path.with_name(npz_path.stem + '_backup.npz')
Path(backup).write_bytes(Path(npz_path).read_bytes())

out = npz_path.with_name(npz_path.stem + '_modified.npz')
save_dict = {k: F[k] for k in F.files}
save_dict[main_key] = data4
save_dict['years'] = years
save_dict['types'] = types
np.savez_compressed(out, **save_dict)
print("Saved modified Floor_area to", out)
# ...existing code...

original data.shape= (501, 501, 4) years= (501,)
expanded data4.shape= (501, 501, 4, 501)
Saved modified Floor_area to Floor_area_TCJ_modified.npz
